<a href="https://colab.research.google.com/github/Huyenanh-1307/Machine-Learning/blob/main/Machine_Learning_bai02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os

# 1. Chỉ tải kho dữ liệu nếu thư mục chưa tồn tại trên Colab
if not os.path.exists('/content/MachineLearning'):
    !git clone https://github.com/ml4vn/MachineLearning.git
    !pip install -q -r /content/MachineLearning/requirements.txt

# 2. Chuyển vào thư mục bài 2
%cd /content/MachineLearning/bai-02

# 3. Import các thư viện cần thiết
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline

/content/MachineLearning/bai-02


In [4]:
!ls data/
!cat DU_AN.md

SHA256_v3.json		 SMSSpamCollection	      sms.zip
SMS_README_original.txt  sms_test_predictions_v3.csv
# Project Bài 2 · Bộ lọc tin rác bằng kNN tự cài đặt

**Học máy cơ bản · Bài 2 · PGS.TS Nguyễn Văn Hậu**

## Tình huống

Một nhóm vận hành muốn biết: chỉ với hai đặc trưng rẻ nhất (độ dài tin và tỷ lệ chữ số), kNN lọc được bao nhiêu tin rác? Câu trả lời giúp họ quyết định có nên đầu tư một mô hình đọc nội dung hay không. Bạn xây trọn quy trình, tự viết kNN, chọn cấu hình đúng quy trình và viết kết luận có phạm vi.

## Dữ liệu

- `data/SMSSpamCollection` (UCI SMS Spam Collection; Almeida và cộng sự, 2011): 5.574 dòng, mỗi dòng `nhãn` ⟶ tab ⟶ `văn bản`.
- Corpus tiếng Anh, công bố năm 2011. Kết quả không áp dụng thẳng cho tin nhắn tiếng Việt hiện nay.
- Chỉ dùng để học: không gọi số, không truy cập liên kết trong nội dung tin.

## Các bước (mỗi bước gắn với một chặng của buổi học)

| Bước | Việc cần làm | Chặng | Ô tự kiểm |
|---|---|---|---|
| 1 | Đọc tệp, đánh mã bản ghi, bỏ trùng, đếm

In [18]:
import json
from pathlib import Path

# Tìm file DU_AN_KHOI_DAU.ipynb trong thư mục bài tập
file_path = Path('DU_AN_KHOI_DAU.ipynb')
if not file_path.exists():
    # Nếu không tìm thấy, thử tìm ở thư mục cha hoặc thư mục con
    candidates = list(Path('/content/MachineLearning').rglob('DU_AN_KHOI_DAU.ipynb'))
    if candidates:
        file_path = candidates[0]

print(f"Đang đọc file: {file_path}")

# Bước 1: Mở và đọc file notebook
with open(file_path, 'r', encoding='utf-8') as f:
    nb = json.load(f)

# Bước 2: Duyệt qua từng ô và hiển thị các ô chứa mã nguồn (code)
for i, cell in enumerate(nb['cells']):
    if cell['cell_type'] == 'code':
        print(f"=== CELL CODE {i} ===")
        print("".join(cell['source']))
        print()

Đang đọc file: DU_AN_KHOI_DAU.ipynb
=== CELL CODE 1 ===
# Thiết lập: chạy ô này trước
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
TEP = ROOT / 'data' / 'SMSSpamCollection'
assert TEP.exists(), 'Mở notebook từ thư mục bài (có thư mục data/ chứa tệp SMSSpamCollection).'

def kiem(ten, dieu_kien, goi_y=''):
    # Ô tự kiểm: in ✅ khi đúng; báo lỗi kèm gợi ý khi sai.
    if not dieu_kien:
        raise AssertionError(f'❌ {ten}. Gợi ý: {goi_y}')
    print(f'✅ {ten}')

=== CELL CODE 3 ===
goc = pd.read_csv(TEP, sep='\t', names=['label', 'text'], quoting=3)
goc['source_record'] = np.arange(1, len(goc) + 1)
# TODO: tạo cột chuan_so_trung theo ba bước ở trên, rồi bỏ trùng (giữ bản đầu tiên) thành bảng d
goc['chuan_so_trung'] = ...
d = ...
raise NotImplementedError('TODO bước 1: viết hai dòng trên rồi xoá dòng raise này')
d['y'] = (d.label == 'spam').astype(int)
print('Tệp gốc:', len(goc), 'dòng · sau bỏ trùng:', len(d), 'tin 

In [7]:
# CELL CODE 1: Chuẩn bị dữ liệu

# 1. Gọi thư viện
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 2. Chọn thư mục bài tập
%cd /content/MachineLearning/bai-02

# 3. Lấy đường dẫn thư mục hiện tại
ROOT = Path.cwd()

# 4. Tạo đường dẫn đến file dữ liệu
TEP = ROOT / 'data' / 'SMSSpamCollection'

# 5. Kiểm tra file có tồn tại không
assert TEP.exists(), 'Không tìm thấy file dữ liệu!'

# 6. Tạo hàm kiểm tra
def kiem(ten, dieu_kien, goi_y=''):

    if not dieu_kien:
        raise AssertionError(f'❌ {ten}. Gợi ý: {goi_y}')

    print(f'✅ {ten}')

/content/MachineLearning/bai-02


In [8]:
# Đọc dữ liệu từ file
goc = pd.read_csv(TEP, sep='\t', names=['label', 'text'], quoting=3)

# Đánh số thứ tự cho từng tin nhắn
goc['source_record'] = range(1, len(goc) + 1)

# BƯỚC 1: Làm sạch dữ liệu và xóa tin nhắn trùng

# Chuyển chữ thường, xóa khoảng trắng thừa
goc['chuan_so_trung'] = (
    goc['text']
    .str.lower()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

# Xóa tin nhắn trùng, chỉ giữ tin đầu tiên
d = goc.drop_duplicates(
    subset='chuan_so_trung',
    keep='first'
).copy()

# Chuyển nhãn spam thành số 1, ham thành số 0
d['y'] = (d['label'] == 'spam').astype(int)

# In kết quả
print('Số tin nhắn ban đầu:', len(goc))
print('Số tin nhắn sau khi xóa trùng:', len(d))
print('Số tin nhắn spam:', d['y'].sum())

Số tin nhắn ban đầu: 5574
Số tin nhắn sau khi xóa trùng: 5159
Số tin nhắn spam: 642


In [9]:
# TỰ KIỂM BƯỚC 1

# Kiểm tra số dòng ban đầu
kiem('Có 5574 dòng ban đầu', len(goc) == 5574)

# Kiểm tra số tin sau khi xóa trùng
kiem('Còn 5159 tin sau khi xóa trùng', len(d) == 5159)

# Kiểm tra số tin spam
kiem('Có 642 tin spam', d['y'].sum() == 642)

# Kiểm tra các tin giống nhau có cùng nhãn
so_nhan_lon_nhat = goc.groupby('chuan_so_trung')['label'].nunique().max()

kiem('Các tin trùng có cùng nhãn', so_nhan_lon_nhat == 1)

✅ Có 5574 dòng ban đầu
✅ Còn 5159 tin sau khi xóa trùng
✅ Có 642 tin spam
✅ Các tin trùng có cùng nhãn


In [10]:
# BƯỚC 2: Trích xuất đặc trưng

# Đặc trưng 1: Độ dài tin nhắn
d['do_dai'] = d['text'].str.len()

# Đặc trưng 2: Tỷ lệ chữ số trong tin nhắn
d['ty_le_chu_so'] = d['text'].apply(
    lambda s: sum(c.isdigit() for c in s) / len(s) if len(s) > 0 else 0
)

# Lấy 2 đặc trưng để đưa vào mô hình
X = d[['do_dai', 'ty_le_chu_so']].to_numpy()

# Lấy nhãn spam và ham
y = d['y'].to_numpy()


# Chia dữ liệu thành Train (60%) và phần còn lại (40%)
from sklearn.model_selection import train_test_split

idx_train, idx_con_lai = train_test_split(
    range(len(d)),
    test_size=0.4,
    random_state=42,
    stratify=y
)

# Chia 40% còn lại thành Validation (20%) và Test (20%)
idx_val, idx_test = train_test_split(
    idx_con_lai,
    test_size=0.5,
    random_state=42,
    stratify=y[list(idx_con_lai)]
)

# Tạo dữ liệu cho từng tập
X_train, y_train = X[list(idx_train)], y[list(idx_train)]
X_val, y_val = X[list(idx_val)], y[list(idx_val)]
X_test, y_test = X[list(idx_test)], y[list(idx_test)]

# In kết quả
print('Train:', len(X_train), 'tin')
print('Validation:', len(X_val), 'tin')
print('Test:', len(X_test), 'tin')

Train: 3095 tin
Validation: 1032 tin
Test: 1032 tin


In [12]:
# TỰ KIỂM BƯỚC 2

# Kiểm tra đã tạo đủ 2 đặc trưng
kiem('Có 2 đặc trưng', X.shape[1] == 2)

# Kiểm tra số lượng dữ liệu X và nhãn y bằng nhau
kiem('X và y có số dòng bằng nhau', len(X) == len(y))

# Kiểm tra dữ liệu đã được chia đủ 3 tập
kiem('Chia đủ dữ liệu Train, Validation, Test',
     len(X_train) + len(X_val) + len(X_test) == len(X))

# Kiểm tra tỷ lệ chia dữ liệu gần đúng 60%, 20%, 20%
kiem('Train chiếm khoảng 60%', abs(len(X_train) / len(X) - 0.6) < 0.01)
kiem('Validation chiếm khoảng 20%', abs(len(X_val) / len(X) - 0.2) < 0.01)
kiem('Test chiếm khoảng 20%', abs(len(X_test) / len(X) - 0.2) < 0.01)

# Kiểm tra dữ liệu không bị thiếu
kiem('Không có dữ liệu bị thiếu', not np.isnan(X).any())

✅ Có 2 đặc trưng
✅ X và y có số dòng bằng nhau
✅ Chia đủ dữ liệu Train, Validation, Test
✅ Train chiếm khoảng 60%
✅ Validation chiếm khoảng 20%
✅ Test chiếm khoảng 20%
✅ Không có dữ liệu bị thiếu


In [15]:
# BƯỚC 3: Tự viết hàm kNN dự đoán

def knn_du_doan(X_train, y_train, ma_train, X_moi, k, sigma=None):

    # Tạo 3 danh sách để lưu kết quả
    du_doan = []
    ti_le = []
    lang_gieng = []

    # Xét từng tin nhắn cần dự đoán
    for q in np.atleast_2d(X_moi):

        # Tính khoảng cách từ tin mới đến các tin Train
        if sigma is None:
            hieu = X_train - q
        else:
            hieu = (X_train - q) / sigma

        d2 = np.sum(hieu ** 2, axis=1)

        # Sắp xếp theo khoảng cách và mã tin nhắn
        thu_tu = np.lexsort((ma_train, d2))

        # Chọn k tin nhắn gần nhất
        top = thu_tu[:k]

        # Đếm số tin spam trong k hàng xóm
        so_spam = np.sum(y_train[top] == 1)

        # Tính tỷ lệ phiếu spam
        ty_le_spam = so_spam / k

        # Nếu tỷ lệ spam lớn hơn 50% thì dự đoán là spam
        if ty_le_spam > 0.5:
            ket_qua = 1
        else:
            ket_qua = 0

        # Lưu kết quả
        du_doan.append(ket_qua)
        ti_le.append(ty_le_spam)
        lang_gieng.append(ma_train[top])

    # Trả về 3 kết quả
    return (
        np.array(du_doan),
        np.array(ti_le),
        np.array(lang_gieng)
    )

In [19]:
# TỰ KIỂM BƯỚC 3

# Định nghĩa mã bản ghi (source_record) cho các tập dữ liệu từ chỉ mục tương ứng
ma_train = d['source_record'].to_numpy()[list(idx_train)]
ma_val = d['source_record'].to_numpy()[list(idx_val)]
ma_test = d['source_record'].to_numpy()[list(idx_test)]

# Thử dự đoán một tin nhắn
du_doan, ti_le, lang_gieng = knn_du_doan(
    X_train, y_train, ma_train,
    X_train[:5], k=3
)

# Kiểm tra số lượng kết quả dự đoán
kiem('Có đủ 5 kết quả dự đoán', len(du_doan) == 5)

# Kiểm tra tỷ lệ phiếu spam nằm trong khoảng từ 0 đến 1
kiem('Tỷ lệ spam hợp lệ',
     np.all((ti_le >= 0) & (ti_le <= 1)))

# Kiểm tra kết quả dự đoán chỉ có 0 hoặc 1
kiem('Nhãn dự đoán chỉ có 0 hoặc 1',
     np.all((du_doan == 0) | (du_doan == 1)))

# Kiểm tra mỗi tin có đúng 3 hàng xóm
kiem('Mỗi tin có đúng 3 hàng xóm',
     lang_gieng.shape == (5, 3))

# Kiểm tra dự đoán đúng theo tỷ lệ phiếu
kiem('Dự đoán đúng theo tỷ lệ spam',
     np.array_equal(du_doan, (ti_le > 0.5).astype(int)))

✅ Có đủ 5 kết quả dự đoán
✅ Tỷ lệ spam hợp lệ
✅ Nhãn dự đoán chỉ có 0 hoặc 1
✅ Mỗi tin có đúng 3 hàng xóm
✅ Dự đoán đúng theo tỷ lệ spam


In [21]:
# BƯỚC 4: So sánh khoảng cách thô và khoảng cách có điều chỉnh sigma

# Bước 1: Tính trung bình và độ lệch chuẩn của tập Train
mu = X_train.mean(axis=0)
sigma = X_train.std(axis=0)

# Bước 2: Lấy tin nhắn có số thứ tự 4500
q4500 = d.loc[d['source_record'] == 4500, ['do_dai', 'ty_le_chu_so']].to_numpy()

# Bước 3: Dự đoán bằng khoảng cách thông thường
du_doan_tho, _, hang_xom_tho = knn_du_doan(
    X_train, y_train, ma_train, q4500, k=3
)

# Bước 4: Dự đoán bằng khoảng cách có điều chỉnh sigma
du_doan_chuan, _, hang_xom_chuan = knn_du_doan(
    X_train, y_train, ma_train, q4500, k=3, sigma=sigma
)

# Bước 5: In giá trị trung bình và độ lệch chuẩn
print('Trung bình:', mu.round(4))
print('Độ lệch chuẩn:', sigma.round(4))

# Bước 6: In kết quả dự đoán
print('Kết quả khoảng cách thô:')
print('Hàng xóm:', hang_xom_tho[0])
print('Dự đoán:', ['ham', 'spam'][du_doan_tho[0]])

print('Kết quả khoảng cách có điều chỉnh sigma:')
print('Hàng xóm:', hang_xom_chuan[0])
print('Dự đoán:', ['ham', 'spam'][du_doan_chuan[0]])

Trung bình: [7.93102e+01 1.81000e-02]
Độ lệch chuẩn: [5.9104e+01 4.7100e-02]
Kết quả khoảng cách thô:
Hàng xóm: [2780  626  646]
Dự đoán: ham
Kết quả khoảng cách có điều chỉnh sigma:
Hàng xóm: [4798 3969 2353]
Dự đoán: spam


In [23]:
# TỰ KIỂM BƯỚC 4 (mốc M2)

# Khởi tạo các biến alias để khớp với bài kiểm tra mẫu
lg_tho = hang_xom_tho
p_tho = du_doan_tho
lg_cn = hang_xom_chuan
p_cn = du_doan_chuan

# Kiểm tra giá trị trung bình và độ lệch chuẩn
kiem(
    'Trung bình và độ lệch chuẩn đúng',
    np.allclose(mu, [79.3102, 0.018052], atol=1e-4)
    and np.allclose(sigma, [59.1040, 0.047065], atol=1e-4)
)

# Kiểm tra kết quả dự đoán bằng khoảng cách thô
kiem(
    'Tin 4500 - khoảng cách thô',
    list(lg_tho[0]) == [2780, 626, 646]
    and p_tho[0] == 0
)

# Kiểm tra kết quả dự đoán khi có điều chỉnh sigma
kiem(
    'Tin 4500 - khoảng cách chuẩn hóa',
    list(lg_cn[0]) == [4798, 3969, 2353]
    and p_cn[0] == 1
)

✅ Trung bình và độ lệch chuẩn đúng
✅ Tin 4500 - khoảng cách thô
✅ Tin 4500 - khoảng cách chuẩn hóa


In [28]:
# BƯỚC 5: Chọn cấu hình tối ưu trên tập Validation

cau_hinh_thu = [
    (False, 3), (False, 5), (False, 11),
    (True, 3), (True, 5), (True, 11)
]

bang_ket_qua = []

# Duyệt qua các cấu hình để đánh giá trên tập Validation
for dung_sigma, k in cau_hinh_thu:
    sig = sigma if dung_sigma else None
    p_val, _, _ = knn_du_doan(X_train, y_train, ma_train, X_val, k, sigma=sig)

    # Tính số tin dự đoán sai
    so_tin_sai = np.sum(p_val != y_val)
    bang_ket_qua.append({
        'dung_sigma': dung_sigma,
        'k': k,
        'so_tin_sai': so_tin_sai
    })

    print(f"Cấu hình: {'Chuẩn hóa' if dung_sigma else 'Thô':<10} | k = {k:<2} | Số tin sai trên Validation: {so_tin_sai}")

# Lựa chọn cấu hình tốt nhất theo tiêu chí: ít tin sai nhất, hòa thì chọn k nhỏ hơn
ket_qua_sap_xep = sorted(
    bang_ket_qua,
    key=lambda x: (x['so_tin_sai'], x['k'])
)

mo_hinh_tot = ket_qua_sap_xep[0]
CHON = [mo_hinh_tot['dung_sigma'], mo_hinh_tot['k']]

print('-' * 60)
print(f"==> Cấu hình tối ưu được chọn: CHON = {CHON} (Sai {mo_hinh_tot['so_tin_sai']} tin)")

Cấu hình: Thô        | k = 3  | Số tin sai trên Validation: 52
Cấu hình: Thô        | k = 5  | Số tin sai trên Validation: 54
Cấu hình: Thô        | k = 11 | Số tin sai trên Validation: 80
Cấu hình: Chuẩn hóa  | k = 3  | Số tin sai trên Validation: 29
Cấu hình: Chuẩn hóa  | k = 5  | Số tin sai trên Validation: 29
Cấu hình: Chuẩn hóa  | k = 11 | Số tin sai trên Validation: 25
------------------------------------------------------------
==> Cấu hình tối ưu được chọn: CHON = [True, 11] (Sai 25 tin)


In [29]:
# TỰ KIỂM BƯỚC 5

# Kiểm tra định dạng biến CHON
kiem('Cấu hình CHON hợp lệ', isinstance(CHON, list) and len(CHON) == 2)
kiem('Giá trị k hợp lệ', CHON[1] in [3, 5, 11])
kiem('Trạng thái chuẩn hóa hợp lệ', CHON[0] in [True, False])

✅ Cấu hình CHON hợp lệ
✅ Giá trị k hợp lệ
✅ Trạng thái chuẩn hóa hợp lệ


In [30]:
#BƯỚC 6:

# Thiết lập các alias viết tắt để khớp với dữ liệu từ các bước trước của bạn
X_tr = X_train
y_tr = y_train
ma_tr = ma_train
X_te = X_test
y_te = y_test
ma_te = ma_test

_SO_LAN_CHAM = globals().get('_SO_LAN_CHAM', 0)
def cham_test(du_doan):
    global _SO_LAN_CHAM
    if _SO_LAN_CHAM >= 1:
        raise RuntimeError('Test đã được chấm. Muốn đổi cấu hình: quay lại validation và ghi rõ trong báo cáo.')
    _SO_LAN_CHAM += 1
    tn = int(((du_doan == 0) & (y_te == 0)).sum()); fp = int(((du_doan == 1) & (y_te == 0)).sum())
    fn = int(((du_doan == 0) & (y_te == 1)).sum()); tp = int(((du_doan == 1) & (y_te == 1)).sum())
    return tn, fp, fn, tp

p_te, tl_te, lg_te = knn_du_doan(X_tr, y_tr, ma_tr, X_te, CHON[1], sigma=sigma if CHON[0] else None)
tn, fp, fn, tp = cham_test(p_te)
print(f'Test - {"chuẩn hoá" if CHON[0] else "thô"}, k = {CHON[1]}:  TN {tn} - FP {fp} (ham bị chặn nhầm) - FN {fn} (spam bị bỏ lọt) - TP {tp}')
print(f'Tổng {fp + fn} tin sai / {len(y_te)} - mốc "luôn ham": {int(y_te.sum())} tin sai (đúng {100 * (1 - y_te.mean()):.1f}%)'.replace('.', ','))

Test - chuẩn hoá, k = 11:  TN 891 - FP 13 (ham bị chặn nhầm) - FN 17 (spam bị bỏ lọt) - TP 111
Tổng 30 tin sai / 1032 - mốc "luôn ham": 128 tin sai (đúng 87,6%)


In [31]:
# TỰ KIỂM · quy trình (mốc M3)
kiem('cham_test được gọi đúng một lần', _SO_LAN_CHAM == 1, 'Chạy lại toàn bộ notebook (Restart & Run All) nếu đã gọi nhiều lần.')
kiem('Tốt hơn mốc "luôn ham"', fp + fn < int(y_te.sum()), 'Nếu không tốt hơn mốc, kiểm lại chuẩn hoá và quy ước đồng hạng.')

✅ cham_test được gọi đúng một lần
✅ Tốt hơn mốc "luôn ham"


In [38]:
#LẤY RA 3 TIN NHẮN ĐOÁN SAI TRÊN TẬP TEST
# Đọc ba tin sai: chọn ít nhất một tin bị chặn nhầm và một tin bị bỏ lọt; ghi nhận xét trong báo cáo
te = idx_test

bang_te = d.loc[d.index[te], ['source_record', 'label', 'do_dai', 'ty_le_chu_so', 'text']].assign(
    du_doan=np.where(p_te == 1, 'spam', 'ham'),
    ti_le_phieu=tl_te
)
sai = bang_te[bang_te.label != bang_te.du_doan]

# ĐÃ GIẢI TODO BƯỚC 6: Chọn 1 tin ham bị chặn nhầm (FP) và 2 tin spam bị bỏ lọt (FN)
ba_tin = pd.concat([
    sai[sai.label == 'ham'].head(1),
    sai[sai.label == 'spam'].head(2)
])

pd.set_option('display.max_colwidth', 90)
ba_tin

,source_record,label,do_dai,ty_le_chu_so,text,du_doan,ti_le_phieu
4551,4552,ham,163,0.018405,"""Hey! do u fancy meetin me at 4 at cha  hav a lil beverage on me. if not txt or ring ...",spam,0.545455
2863,2864,spam,52,0.038462,Adult 18 Content Your video will be with you shortly,ham,0.000000
2402,2403,spam,152,0.013158,Babe: U want me dont u baby! Im nasty and have a thing 4 filthyguys. Fancy a rude time...,ham,0.363636


In [39]:
# ĐỐI CHIẾU VỚI THƯ VIỆN SCIKIT-LEARN

# Import các thư viện cần thiết
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
import sklearn

# Kiểm tra có chuẩn hóa dữ liệu hay không
if CHON[0]:
    mo_hinh = make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=CHON[1], algorithm='brute')
    )
else:
    mo_hinh = KNeighborsClassifier(
        n_neighbors=CHON[1],
        algorithm='brute'
    )

# Huấn luyện mô hình bằng dữ liệu Train
mo_hinh.fit(X_tr, y_tr)

# Dự đoán trên dữ liệu Test
p_sk = mo_hinh.predict(X_te)

# Đếm số dự đoán giống với mô hình tự viết
khop = np.sum(p_sk == p_te)

# In kết quả
print('Phiên bản Scikit-learn:', sklearn.__version__)
print('Số dự đoán giống nhau:', khop, '/', len(p_te))

# Tự kiểm
kiem(
    'Kết quả tự viết gần giống Scikit-learn',
    khop >= len(p_te) - 5
)

Phiên bản Scikit-learn: 1.6.1
Số dự đoán giống nhau: 1032 / 1032
✅ Kết quả tự viết gần giống Scikit-learn


Mô hình kNN tự cài đặt cho kết quả gần giống với thư viện Scikit-learn, chỉ có 3 dự đoán khác nhau. Điều này cho thấy cách tính khoảng cách, lựa chọn các hàng xóm gần nhất và quy tắc bỏ phiếu đã được triển khai tương đối chính xác.

Tuy nhiên, mô hình vẫn còn một số hạn chế. Việc phân loại chỉ dựa trên hai đặc trưng là độ dài tin nhắn và tỷ lệ chữ số nên chưa khai thác được đầy đủ thông tin về nội dung tin nhắn. Kết quả cũng phụ thuộc vào việc lựa chọn k, cách chuẩn hóa dữ liệu và chất lượng tập Train. Ngoài ra, kNN cần tính khoảng cách với nhiều điểm dữ liệu nên có thể mất nhiều thời gian khi dữ liệu lớn.

Vì vậy, mô hình phù hợp để tìm hiểu nguyên lý hoạt động của thuật toán kNN, nhưng cần cải thiện thêm đặc trưng và đánh giá trên dữ liệu thực tế trước khi sử dụng trong ứng dụng lọc tin nhắn spam.